In [6]:
import json

def load_template(file_path: str) -> str:
    with open(file_path, encoding='utf8') as f:
        data = f.read()
    return data


# The instances must be improved inline, because the experiments differ between 1.6 and <1.6
# We basically only need to adjust the prompts and add the new information
player_a_prompt_header = load_template(f"../resources/initial_prompts/player_a_prompt_header_zero_shot.template")
player_b_prompt_header = load_template(f"../resources/initial_prompts/player_b_prompt_header_zero_shot.template")

def to_list_of_targets(target):
    if target == "first":
        return ["first", "1st", "1"]
    if target == "second":
        return ["second", "2nd", "2"]
    if target == "third":
        return ["third", "3rd", "3"]
    
def migrate(idx, game_instance):
    game_instance["game_id"] = idx
    game_instance["target_grid_name"] = to_list_of_targets(game_instance["target_grid_name"])
    game_instance["player_1_prompt_header"] = player_a_prompt_header.replace('TARGET_GRID', game_instance["player_1_target_grid"])\
                                                                    .replace('SECOND_GRID', game_instance["player_1_second_grid"])\
                                                                    .replace('THIRD_GRID', game_instance["player_1_third_grid"])
    game_instance["player_2_prompt_header"] = player_b_prompt_header.replace('FIRST_GRID', game_instance["player_2_first_grid"])\
                                                                .replace('SECOND_GRID', game_instance["player_2_second_grid"])\
                                                                .replace('THIRD_GRID', game_instance["player_2_third_grid"])
    game_instance['player_1_response_pattern'] = '^expression:\s(?P<content>.+)\n*(?P<remainder>.*)'
    game_instance['player_2_response_pattern'] = '^answer:\s(?P<content>first|second|third|1|2|3|1st|2nd|3rd)\n*(?P<remainder>.*)'
    game_instance["player_1_response_tag"] = "expression:"
    game_instance["player_2_response_tag"] = "answer:"

for v in ["0.9", "1.0"]:
    fn = f"instances_v{v}.json"
    with open(fn, encoding="utf-8") as f:
        data = json.load(f)
        
    for experiment in data["experiments"]:
        for idx, game in enumerate(experiment["game_instances"]):
            migrate(idx, game)
            
    with open(fn, encoding="utf-8", mode="w") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        